# 05.2 Model Comparison: Baseline vs PriceSequenceGRU

Comparación reproducible entre `MarketValueNet` y `PriceSequenceGRU`.

**Objetivo**
- evaluar ambos modelos en el mismo subconjunto temporal de validación
- comparar cobertura, métricas y desacuerdos
- revisar overlap de oportunidades en mercados activos
- decidir si el modelo secuencial aporta señal nueva o solo replica al baseline

**Prerequisitos**
- `data/models/best_market_model.pt`
- `data/models/ts_gru/best_ts_gru_model.pt`
- `data/processed/pipeline/`
- `data/processed_ts/`

## 1. Setup

Cargamos raw data, checkpoints y pipeline del baseline. Si alguno de estos artefactos falta, la comparación todavía no está lista.

In [ ]:
import sys
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.features.pipeline import FeaturePipeline
from src.model.architecture import MarketValueNet
from src.model.ts_architecture import PriceSequenceGRU
from src.model.ts_dataset import TimeSeriesMarketDataset
from src.scoring.scorer import score_active_markets
from src.scoring.ts_scorer import score_active_markets_ts, score_resolved_markets_ts
from src.scoring.signals import generate_signals
from src.data.preprocessing import build_snapshot_market, _infer_resolution_from_market

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

cfg = load_config(str(ROOT / 'config' / 'config.yaml'))
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / cfg['data']['processed_dir']
PROCESSED_TS = ROOT / cfg['ts_data']['processed_dir']
FIGURES = ROOT / 'figures'

baseline_model_path = ROOT / 'data' / 'models' / 'best_market_model.pt'
ts_model_path = ROOT / cfg['ts_training']['save_dir'] / 'best_ts_gru_model.pt'
pipeline_dir = PROCESSED / 'pipeline'

assert baseline_model_path.exists(), f'Falta modelo baseline: {baseline_model_path}'
assert ts_model_path.exists(), f'Falta modelo TS: {ts_model_path}'
assert pipeline_dir.exists(), f'Falta pipeline baseline: {pipeline_dir}'
assert PROCESSED_TS.exists(), f'Falta dataset TS: {PROCESSED_TS}'


## 2. Definición del universo común de evaluación

Tomamos como referencia el split temporal del modelo TS y reconstruimos ese mismo subconjunto de validación para poder comparar de forma limpia.

In [ ]:
with open(RAW / 'resolved_markets.json') as f:
    resolved_markets = json.load(f)
with open(RAW / 'active_markets.json') as f:
    active_markets = json.load(f)
with open(RAW / 'price_histories.json') as f:
    price_histories = json.load(f)
with open(RAW / 'order_books.json') as f:
    order_books = json.load(f)

resolved_by_id = {str(m['id']): m for m in resolved_markets}

ts_dataset = TimeSeriesMarketDataset.from_numpy_dir(str(PROCESSED_TS))
n = len(ts_dataset)
n_val = int(n * cfg['training']['val_split'])
n_train = n - n_val
sorted_indices = np.argsort(ts_dataset.timestamps)
val_indices = sorted_indices[n_train:]
val_market_ids = [str(ts_dataset.market_ids[i]) for i in val_indices]
val_markets = [resolved_by_id[mid] for mid in val_market_ids if mid in resolved_by_id]

print(f'Mercados val TS: {len(val_markets):,}')
print(f'Mercados activos cacheados: {len(active_markets):,}')


## 3. Carga de ambos modelos

Aquí no entrenamos nada: solo cargamos los checkpoints ya producidos por `04` y `04_1`.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

baseline_pipeline = FeaturePipeline.load(str(pipeline_dir), use_dummy_text=False)
baseline_model = MarketValueNet(
    num_numerical_features=baseline_pipeline.num_numerical_features,
    num_categories=baseline_pipeline.num_categories,
    text_embed_dim=baseline_pipeline.text_embed_dim,
    hidden_dims=cfg['model']['hidden_dims'],
    dropout=cfg['model']['dropout'],
    task='classification',
)
baseline_model.load_state_dict(torch.load(baseline_model_path, map_location='cpu', weights_only=True))
baseline_model = baseline_model.to(device).eval()

ts_model = PriceSequenceGRU(
    input_dim=cfg['ts_model']['input_dim'],
    hidden_dim=cfg['ts_model']['hidden_dim'],
    num_layers=cfg['ts_model']['num_layers'],
    dropout=cfg['ts_model']['dropout'],
    task='classification',
)
ts_model.load_state_dict(torch.load(ts_model_path, map_location='cpu', weights_only=True))
ts_model = ts_model.to(device).eval()


## 4. Scoring del subconjunto temporal compartido

`PriceSequenceGRU` y `MarketValueNet` se evalúan sobre los mismos mercados resueltos y el mismo cutoff temporal. La intersección exacta es la base de la comparación offline.

In [ ]:
df_ts_val = score_resolved_markets_ts(
    ts_model,
    val_markets,
    price_histories=price_histories,
    snapshot_offset_days=cfg['ts_data']['snapshot_offset_days'],
    seq_len=cfg['ts_data']['seq_len'],
    min_points=cfg['ts_data']['min_points'],
    device=device,
)

baseline_rows = []
skipped_baseline = {'no_snapshot': 0, 'ambiguous': 0, 'exceptions': 0}
for market in val_markets:
    market_id = str(market.get('id', ''))
    history = price_histories.get(market_id)
    market_snapshot, snapshot_price = build_snapshot_market(
        market,
        price_histories=price_histories,
        snapshot_offset_days=cfg['ts_data']['snapshot_offset_days'],
    )
    if market_snapshot is None or snapshot_price is None:
        skipped_baseline['no_snapshot'] += 1
        continue

    resolution = _infer_resolution_from_market(market)
    if resolution not in ('yes', 'no'):
        skipped_baseline['ambiguous'] += 1
        continue

    try:
        features = baseline_pipeline.transform_single(market_snapshot, price_history=history)
        num_t = torch.FloatTensor(features['numerical']).unsqueeze(0).to(device)
        cat_t = torch.LongTensor([features['category_id']]).to(device)
        txt_t = torch.FloatTensor(features['text_embedding']).unsqueeze(0).to(device)
        with torch.no_grad():
            score = baseline_model(num_t, cat_t, txt_t).item()
        baseline_rows.append({
            'id': market_id,
            'question': market.get('question', ''),
            'price_yes': float(snapshot_price),
            'volume_24h': float(market.get('volume24hr', 0) or 0),
            'liquidity': float(market.get('liquidity', 0) or 0),
            'spread': float(market.get('spread', 0) or 0),
            'model_score': score,
            'resolved': 1 if resolution == 'yes' else 0,
            'slug': market.get('slug', ''),
        })
    except Exception:
        skipped_baseline['exceptions'] += 1

df_baseline_val = pd.DataFrame(baseline_rows)
print('Skipped baseline val:', skipped_baseline)
print(f'TS val scored:       {len(df_ts_val):,}')
print(f'Baseline val scored: {len(df_baseline_val):,}')


## 5. Métricas comparables en validación

Calculamos `AUC-ROC`, `PR-AUC` y `accuracy` para ambos modelos sobre la misma intersección de mercados válidos.

In [ ]:
df_cmp = df_baseline_val.merge(
    df_ts_val[['id', 'model_score', 'sequence_length', 'resolved']],
    on=['id', 'resolved'],
    suffixes=('_baseline', '_ts'),
)
print(f'Intersección exacta en validación: {len(df_cmp):,} mercados')

def summarize_model(name, labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)
    preds = (scores > 0.5).astype(int)
    if len(np.unique(labels)) >= 2:
        auc_roc = roc_auc_score(labels, scores)
        pr_auc = average_precision_score(labels, scores)
    else:
        auc_roc = 0.0
        pr_auc = 0.0
    acc = float((preds == labels).mean())
    return {'model': name, 'auc_roc': auc_roc, 'pr_auc': pr_auc, 'accuracy': acc}

metrics_df = pd.DataFrame([
    summarize_model('MarketValueNet', df_cmp['resolved'], df_cmp['model_score_baseline']),
    summarize_model('PriceSequenceGRU', df_cmp['resolved'], df_cmp['model_score_ts']),
])
display(metrics_df.round(4))


## 6. Lectura visual de desempeño offline

Este bloque ayuda a responder dos preguntas:
- cuál modelo se ve mejor en las métricas básicas
- cuánto se parecen realmente sus scores mercado por mercado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
metric_plot = metrics_df.set_index('model')[['auc_roc', 'pr_auc', 'accuracy']]
metric_plot.plot.bar(ax=ax, color=[PALETTE[0], PALETTE[1], PALETTE[2]], alpha=0.85)
ax.set_title('Validación temporal — métricas')
ax.set_ylim(0, 1)
ax.legend(loc='lower right')

ax = axes[1]
ax.scatter(df_cmp['model_score_baseline'], df_cmp['model_score_ts'], alpha=0.3, s=12, color=PALETTE[0])
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_title('Score baseline vs score TS')
ax.set_xlabel('MarketValueNet score')
ax.set_ylabel('PriceSequenceGRU score')

plt.tight_layout()
plt.savefig(FIGURES / 'model_comparison_metrics.png', bbox_inches='tight')
plt.show()


## 7. Comparación en mercados activos

Después de la comparación offline, llevamos ambos modelos al mismo snapshot local de activos y aplicamos las mismas reglas de señales. Eso permite medir overlap real de ideas operativas.

In [ ]:
class LocalClient:
    def __init__(self, markets):
        self.markets = markets
    def get_all_active_markets(self, max_markets=1000):
        return self.markets[:max_markets]
    def parse_market(self, market):
        return market

local_client = LocalClient(active_markets)
df_active_baseline = score_active_markets(
    baseline_model,
    local_client,
    baseline_pipeline,
    price_histories=price_histories,
    order_books=order_books,
    fetch_missing_context=False,
    top_k=len(active_markets),
    max_markets=len(active_markets),
)
df_active_ts = score_active_markets_ts(
    ts_model,
    local_client,
    price_histories=price_histories,
    fetch_missing_history=False,
    top_k=len(active_markets),
    max_markets=len(active_markets),
    seq_len=cfg['ts_data']['seq_len'],
    min_points=cfg['ts_data']['min_points'],
)

df_active_baseline = generate_signals(
    df_active_baseline,
    buy_threshold=cfg['scoring']['buy_threshold'],
    strong_buy_threshold=cfg['scoring']['strong_buy_threshold'],
    min_liquidity=cfg['scoring']['min_liquidity'],
    min_volume_24h=cfg['scoring']['min_volume_24h'],
    max_spread=cfg['scoring']['max_spread'],
)
df_active_ts = generate_signals(
    df_active_ts,
    buy_threshold=cfg['scoring']['buy_threshold'],
    strong_buy_threshold=cfg['scoring']['strong_buy_threshold'],
    min_liquidity=cfg['scoring']['min_liquidity'],
    min_volume_24h=cfg['scoring']['min_volume_24h'],
    max_spread=cfg['scoring']['max_spread'],
)

top_k = 20
top_base_ids = set(df_active_baseline.head(top_k)['id'])
top_ts_ids = set(df_active_ts.head(top_k)['id'])
overlap = len(top_base_ids & top_ts_ids)
print(f'Activos puntuados baseline: {len(df_active_baseline):,}')
print(f'Activos puntuados TS:       {len(df_active_ts):,}')
print(f'Overlap top-{top_k}:        {overlap} mercados')


## 8. Overlap y desacuerdos más fuertes

La parte más útil de esta sección no es solo el overlap, sino los mercados donde ambos modelos discrepan mucho. Ahí suele estar la señal nueva del modelo secuencial.

In [ ]:
overlap_df = pd.DataFrame({
    'bucket': ['Solo baseline', 'Overlap', 'Solo TS'],
    'count': [len(top_base_ids - top_ts_ids), len(top_base_ids & top_ts_ids), len(top_ts_ids - top_base_ids)],
})

active_compare = df_active_baseline[['id', 'question', 'model_score', 'signal']].merge(
    df_active_ts[['id', 'model_score', 'signal']],
    on='id',
    suffixes=('_baseline', '_ts'),
)
active_compare['score_gap'] = active_compare['model_score_ts'] - active_compare['model_score_baseline']
disagreements = active_compare.sort_values('score_gap', key=np.abs, ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(overlap_df['bucket'], overlap_df['count'], color=[PALETTE[0], PALETTE[2], PALETTE[1]], alpha=0.85)
ax.set_title('Overlap de top-20 en activos')
ax.set_ylabel('Mercados')

ax = axes[1]
ax.hist(active_compare['score_gap'], bins=30, color=PALETTE[0], alpha=0.8)
ax.axvline(0, color='black', linestyle='--', lw=1.5)
ax.set_title('TS score - baseline score')
ax.set_xlabel('Diferencia de score')

plt.tight_layout()
plt.savefig(FIGURES / 'model_comparison_overlap.png', bbox_inches='tight')
plt.show()

print('Desacuerdos más fuertes en activos:')
display(disagreements[['id', 'question', 'signal_baseline', 'signal_ts', 'model_score_baseline', 'model_score_ts', 'score_gap']])


## 9. Cómo interpretar el resultado final

Al cerrar este notebook, una lectura útil es:
- si el GRU mejora métricas offline o no
- si propone mercados distintos al baseline en activos
- si las discrepancias parecen razonables según la trayectoria del precio

Si ambos modelos aportan algo distinto, ya tienes base para explorar ensemble o reglas de consenso más adelante.